# One-time scene cache and CNN chips

Implements section 6 of [`PLAN.md`](PLAN.md). Every xView3-SAR validation scene in the link file is downloaded **once**. For each scene this notebook saves, in one Google Drive folder:

- `cache/<scene>.npz`: the 6000 × 6000 benchmark window of VH and VV (dB, float16), the water mask, the wind-speed window (native resolution) and the window's position and georeferencing. Later stages (YOLO, coastline, wind analysis) read this instead of downloading the scene again.
- `chips/<scene>.npz` and `chips_index.csv`: the CFAR candidates of the plan (Hollow OS-CFAR, T = 5.0 dB), one 64 × 64 VH + VV chip per candidate (from the unfiltered images) and its label.

Every candidate count is checked against the 49-scene benchmark results (`multiscene_results.csv`, Hollow OS-CFAR at T = 5.0). The run is resumable: scenes whose files already exist in the Drive folder are skipped. Signed links are never printed or saved. Only the Drive folder `MyDrive/xview3_multiscene_results` is written to; nothing in Drive is deleted.


In [ ]:
!pip -q install rasterio scipy pandas

## 1. Locked configuration and helper functions (same as the benchmark)

In [ ]:
# Same configuration and helper functions as cfar_xview3_benchmark.ipynb (locked).
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import matplotlib.pyplot as plt
from scipy.ndimage import (
    median_filter, percentile_filter, uniform_filter,
    label, find_objects, binary_closing,
)
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment

# ---------------------------------------------------------------------
# Locked configuration. Thresholds were chosen on scene 1 (calibration).
# ---------------------------------------------------------------------
CONFIG = {
    "data_dir": ".",                       # folder that contains one sub-folder per scene id
    "csv_path": "./validation.csv",
    "calibration_scene": "590dd08f71056cacv",   # scene 1: thresholds chosen here
    "test_scene": "b1844cde847a3942v",          # scene 2: run once with locked thresholds
    "crop_size": 6000,                     # window (pixels) centred on the median vessel position
    "nan_fill_db": -30.0,                  # value used for missing SAR pixels
    "brightness_floor_db": -30.0,          # absolute brightness cut-off (effectively off)
    "median_filter_size": 3,
    "closing_kernel": (5, 5),              # note: closing slightly erodes objects touching the window edge
    "min_cluster_area": 3,                 # minimum bounding-box area in pixels
    "match_distance_px": 20.0,             # matching radius (20 px = 200 m)

    "ca_guard_win": 9,
    "ca_bg_win": 25,
    "ca_locked_T_db": 11.6,                # scene-1 optimum was the grid point 11.571

    "os_guard_win": 9,
    "os_outer_win": 21,
    "os_percentile": 85,
    "os_locked_T_db": 8.1,                 # scene-1 optimum was the grid point 8.143

    "twoparam_locked_T_std": 5.5,
    "ref_low_T_db": 5.0,                   # low-threshold, high-recall reference point

    "db_sweep_grid": np.linspace(3.0, 15.0, 15),
    "std_sweep_grid": np.linspace(2.0, 9.0, 15),
}


def prepare_scene(df, scene_id, cfg):
    """Load a window of the scene, build detector background maps and ground-truth lists."""
    scene_dir = os.path.join(cfg["data_dir"], scene_id)
    rows = df[df["scene_id"] == scene_id].copy()
    vessels = rows[rows["is_vessel"] == True]
    med_r = int(vessels["detect_scene_row"].median())
    med_c = int(vessels["detect_scene_column"].median())
    crop = cfg["crop_size"]

    with rasterio.open(os.path.join(scene_dir, "VH_dB.tif")) as src:
        r_off = max(0, min(src.height - crop, med_r - crop // 2))
        c_off = max(0, min(src.width - crop, med_c - crop // 2))
        win = Window(c_off, r_off, crop, crop)
        vh_db = np.nan_to_num(src.read(1, window=win).astype(np.float32), nan=cfg["nan_fill_db"])
        full_w, full_h = src.width, src.height

    with rasterio.open(os.path.join(scene_dir, "owiMask.tif")) as owi:
        sx, sy = owi.width / full_w, owi.height / full_h
        owin = Window(c_off * sx, r_off * sy, crop * sx, crop * sy)
        water_mask = owi.read(1, window=owin, out_shape=(crop, crop)) == 0

    in_win = rows[
        (rows["detect_scene_row"] >= r_off) & (rows["detect_scene_row"] < r_off + crop) &
        (rows["detect_scene_column"] >= c_off) & (rows["detect_scene_column"] < c_off + crop)
    ].copy().reset_index(drop=True)
    in_win["local_row"] = in_win["detect_scene_row"] - r_off
    in_win["local_col"] = in_win["detect_scene_column"] - c_off
    ri = in_win["local_row"].astype(int).clip(0, crop - 1).values
    ci = in_win["local_col"].astype(int).clip(0, crop - 1).values
    in_win["in_water_mask"] = water_mask[ri, ci]

    win_vessels = in_win[in_win["is_vessel"] == True]
    water_rec = in_win[in_win["in_water_mask"]].reset_index(drop=True)
    eval_vessels = water_rec[water_rec["is_vessel"] == True].reset_index(drop=True)
    high = eval_vessels[eval_vessels["confidence"] == "HIGH"]

    cols = ["local_row", "local_col"]
    not_vessel = water_rec["is_vessel"] != True
    not_high = water_rec["confidence"] != "HIGH"

    # detector background maps
    vh_clean = median_filter(vh_db, size=cfg["median_filter_size"])
    bg_w, gd_w = cfg["ca_bg_win"], cfg["ca_guard_win"]
    n_bg = bg_w ** 2 - gd_w ** 2
    ca_bg = (uniform_filter(vh_clean, size=bg_w) * bg_w ** 2
             - uniform_filter(vh_clean, size=gd_w) * gd_w ** 2) / n_bg
    sq = (uniform_filter(vh_clean ** 2, size=bg_w) * bg_w ** 2
          - uniform_filter(vh_clean ** 2, size=gd_w) * gd_w ** 2) / n_bg
    ca_std = np.sqrt(np.maximum(sq - ca_bg ** 2, 1e-4))

    out_w, os_gd = cfg["os_outer_win"], cfg["os_guard_win"]
    fp = np.ones((out_w, out_w), dtype=bool)
    pad = (out_w - os_gd) // 2
    fp[pad:pad + os_gd, pad:pad + os_gd] = False
    os_bg = percentile_filter(vh_clean, percentile=cfg["os_percentile"], footprint=fp)

    return {
        "scene_id": scene_id, "vh_clean": vh_clean, "ca_bg": ca_bg, "ca_std": ca_std,
        "os_bg": os_bg, "water_mask": water_mask,
        "n_window": len(win_vessels), "n_masked": int((~win_vessels["in_water_mask"]).sum()),
        "n_eval": len(eval_vessels), "n_high": len(high),
        "gt_all": eval_vessels[cols].values, "ig_all": water_rec[not_vessel][cols].values,
        "gt_high": high[cols].values, "ig_high": water_rec[not_vessel | not_high][cols].values,
    }


def extract_centroids(binary_mask, water_mask, cfg):
    """Closing + connected components; returns one centroid per candidate."""
    clean = binary_closing(binary_mask & water_mask, structure=np.ones(cfg["closing_kernel"]))
    lbl, _ = label(clean)
    pts = []
    for box in find_objects(lbl):
        area = (box[0].stop - box[0].start) * (box[1].stop - box[1].start)
        if area >= cfg["min_cluster_area"]:
            pts.append([(box[0].start + box[0].stop) / 2.0, (box[1].start + box[1].stop) / 2.0])
    return np.array(pts) if pts else np.empty((0, 2))


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return 0.0, 0.0
    p = k / n
    denom = 1.0 + z ** 2 / n
    center = (p + z ** 2 / (2.0 * n)) / denom
    half = z * np.sqrt(p * (1.0 - p) / n + z ** 2 / (4.0 * n ** 2)) / denom
    return max(0.0, center - half), min(1.0, center + half)


def evaluate_predictions(pred, target_gt, ignore_gt, max_dist=20.0):
    """One-to-one matching. Predictions near an 'ignore' object are not counted as false alarms."""
    if len(pred) == 0:
        return 0, 0, len(target_gt), 0.0, 0.0, 0.0
    if len(target_gt) == 0:
        return 0, len(pred), 0, 0.0, 0.0, 0.0
    d = cdist(pred, target_gt)
    p_idx, g_idx = linear_sum_assignment(d)
    matched = {p for p, g in zip(p_idx, g_idx) if d[p, g] <= max_dist}
    rest = [i for i in range(len(pred)) if i not in matched]
    ignored = 0
    if rest and len(ignore_gt) > 0:
        ignored = int((cdist(pred[rest], ignore_gt).min(axis=1) <= max_dist).sum())
    tp, fp = len(matched), len(rest) - ignored
    fn = len(target_gt) - tp
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return tp, fp, fn, prec, rec, f1


def detect(S, algo, T, cfg):
    """Binary detection mask for one algorithm at threshold T."""
    thr = algo["bg"] + (T * algo["std"] if algo["std"] is not None else T)
    return (S["vh_clean"] > thr) & (S["vh_clean"] > cfg["brightness_floor_db"])


def algo_table(S, cfg):
    return {
        "2D CA-CFAR": dict(bg=S["ca_bg"], std=None, grid=cfg["db_sweep_grid"],
                           locked=cfg["ca_locked_T_db"], ref=cfg["ref_low_T_db"], color="tab:red"),
        "Hollow OS-CFAR": dict(bg=S["os_bg"], std=None, grid=cfg["db_sweep_grid"],
                               locked=cfg["os_locked_T_db"], ref=cfg["ref_low_T_db"], color="tab:cyan"),
        "Two-Param CFAR": dict(bg=S["ca_bg"], std=S["ca_std"], grid=cfg["std_sweep_grid"],
                               locked=cfg["twoparam_locked_T_std"], ref=None, color="tab:orange"),
    }


def eval_both(S, pts, cfg):
    md_ = cfg["match_distance_px"]
    return (evaluate_predictions(pts, S["gt_all"], S["ig_all"], md_),
            evaluate_predictions(pts, S["gt_high"], S["ig_high"], md_))


def print_table(title, rows):
    print(title)
    print("-" * 104)
    print(f"{'OPERATING POINT':<36}| {'TP':<4}| {'FP':<5}| {'FN':<4}| {'PRECISION (95% CI)':<22}| {'RECALL (95% CI)':<22}| F1")
    print("-" * 104)
    for name, (tp, fp, fn, prec, rec, f1) in rows:
        pl, ph = wilson_ci(tp, tp + fp)
        rl, rh = wilson_ci(tp, tp + fn)
        print(f"{name:<36}| {tp:<4}| {fp:<5}| {fn:<4}| {prec*100:5.1f}% ({pl*100:4.1f}-{ph*100:4.1f})  "
              f"| {rec*100:5.1f}% ({rl*100:4.1f}-{rh*100:4.1f})  | {f1*100:5.1f}%")
    print()


## 2. Input files

In [ ]:
import os, glob, shutil
LINKS_FILE = "validation.txt"
LABELS_FILE = os.path.basename(CONFIG["csv_path"])   # validation.csv

def normalise_names():
    """Browsers may save downloads as 'validation (1).csv' etc. Copy them to the expected names."""
    for target, pattern in [(LINKS_FILE, "validation*.txt"), (LABELS_FILE, "validation*.csv")]:
        if not os.path.exists(target):
            found = sorted(glob.glob(pattern), key=os.path.getmtime)
            if found:
                shutil.copy(found[-1], target)

normalise_names()
missing = [f for f in [LINKS_FILE, LABELS_FILE] if not os.path.exists(f)]
if missing:
    from google.colab import files
    print("Select these files:", ", ".join(missing))
    files.upload()   # file contents are not printed
    normalise_names()
for f in [LINKS_FILE, LABELS_FILE]:
    print(f, "present:", os.path.exists(f))
assert all(os.path.exists(f) for f in [LINKS_FILE, LABELS_FILE]), "Upload both validation.txt and validation.csv, then run again."


## 3. Google Drive folder

In [ ]:
# Results go to one Google Drive folder only. Nothing in Drive is deleted.
import os, shutil
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/xview3_multiscene_results"
CACHE_DIR = os.path.join(DRIVE_DIR, "cache")
CHIPS_DIR = os.path.join(DRIVE_DIR, "chips")
CHIPS_INDEX = os.path.join(DRIVE_DIR, "chips_index.csv")
BENCH_RESULTS = os.path.join(DRIVE_DIR, "multiscene_results.csv")
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(CHIPS_DIR, exist_ok=True)
print("Drive folder:", DRIVE_DIR, "| benchmark results available:", os.path.exists(BENCH_RESULTS))


## 4. Cache and chip functions

In [ ]:
import subprocess, time, json
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist

CHIP = 64              # chip size in pixels (plan section 2)
CLIP_DB = (-35.0, 5.0) # dB range mapped to [0, 1] for chips
CAND_T = CONFIG["ref_low_T_db"]  # 5.0 dB, OS-CFAR candidate threshold (plan section 1)

def scene_id_from_url(url):
    return url.split("?")[0].rstrip("/").split("/")[-1].replace(".tar.gz", "")

def download_scene_files(scene_id, url, data_dir, members=("VH_dB.tif", "VV_dB.tif", "owiMask.tif", "owiWindSpeed.tif")):
    """Download one scene and extract only the listed files. The signed link is never printed."""
    tar_path = os.path.join(data_dir, f"{scene_id}.tar.gz")
    try:
        subprocess.run(["wget", "-q", "-O", tar_path, url], check=True)
        subprocess.run(["tar", "-xzf", tar_path, "-C", data_dir, "--wildcards"] + [f"*/{m}" for m in members], check=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"download/extract failed (exit code {e.returncode}); "
                           "the signed links may have expired, get a fresh validation.txt") from None
    finally:
        if os.path.exists(tar_path):
            os.remove(tar_path)

def read_window(path, r_off, c_off, crop):
    with rasterio.open(path) as src:
        return src.read(1, window=Window(c_off, r_off, crop, crop)).astype(np.float32)

def scene_window_offsets(df, scene_id, cfg):
    """Same window rule as prepare_scene (median labelled-vessel position, clipped to the image)."""
    rows = df[(df["scene_id"] == scene_id) & (df["is_vessel"] == True)]
    med_r, med_c = int(rows["detect_scene_row"].median()), int(rows["detect_scene_column"].median())
    with rasterio.open(os.path.join(cfg["data_dir"], scene_id, "VH_dB.tif")) as src:
        crop = cfg["crop_size"]
        r_off = max(0, min(src.height - crop, med_r - crop // 2))
        c_off = max(0, min(src.width - crop, med_c - crop // 2))
        win_transform = rasterio.windows.transform(Window(c_off, r_off, crop, crop), src.transform)
        return r_off, c_off, src.width, src.height, win_transform, (src.crs.to_wkt() if src.crs else "")

def make_chips(vh_db, vv_db, centres):
    """64 x 64 VH + VV chips (unfiltered dB) around each centre, edge-padded with the clip minimum."""
    h = CHIP // 2
    lo, hi = CLIP_DB
    stack = np.stack([np.nan_to_num(vh_db, nan=lo), np.nan_to_num(vv_db, nan=lo)])
    stack = np.pad(np.clip(stack, lo, hi), ((0, 0), (h, h), (h, h)), constant_values=lo)
    out = np.empty((len(centres), 2, CHIP, CHIP), dtype=np.float16)
    for k, (r, c) in enumerate(centres):
        r, c = int(round(r)) + h, int(round(c)) + h
        out[k] = ((stack[:, r - h:r + h, c - h:c + h] - lo) / (hi - lo)).astype(np.float16)
    return out

def label_candidates(pts, S, eval_df, max_dist):
    """Plan section 2: 1 = matched vessel, 0 = clean negative, -1 = excluded from training (with reason)."""
    n = len(pts)
    label = np.zeros(n, dtype=int); reason = [""] * n
    src = [""] * n; conf = [""] * n
    if n and len(S["gt_all"]):
        d = cdist(pts, S["gt_all"]); p_idx, g_idx = linear_sum_assignment(d)
        for p, g in zip(p_idx, g_idx):
            if d[p, g] <= max_dist:
                label[p] = 1; src[p] = str(eval_df.iloc[g]["source"]); conf[p] = str(eval_df.iloc[g]["confidence"])
        near_vessel = d.min(axis=1) <= max_dist
    else:
        near_vessel = np.zeros(n, dtype=bool)
    near_ignore = (cdist(pts, S["ig_all"]).min(axis=1) <= max_dist) if (n and len(S["ig_all"])) else np.zeros(n, dtype=bool)
    for i in range(n):
        if label[i] == 1:
            continue
        if near_ignore[i]:
            label[i] = -1; reason[i] = "near_ignore_object"
        elif near_vessel[i]:
            label[i] = -1; reason[i] = "near_vessel_unmatched"
    return label, reason, src, conf


## 5. Download each scene once, cache it, cut chips (resumable)

In [ ]:
df_val = pd.read_csv(CONFIG["csv_path"])
with open(LINKS_FILE) as f:
    url_by_scene = {scene_id_from_url(l.strip()): l.strip() for l in f if l.strip().startswith("https://")}
scene_ids = sorted(url_by_scene)
bench = pd.read_csv(BENCH_RESULTS) if os.path.exists(BENCH_RESULTS) else None
print(f"Scenes in link file: {len(scene_ids)}")

def done(sid):
    return os.path.exists(os.path.join(CACHE_DIR, f"{sid}.npz")) and os.path.exists(os.path.join(CHIPS_DIR, f"{sid}.npz"))

t_start = time.time()
for i, sid in enumerate(scene_ids, 1):
    if done(sid):
        continue
    t0 = time.time()
    if int(((df_val["scene_id"] == sid) & (df_val["is_vessel"] == True)).sum()) == 0:
        print(f"[{i}/{len(scene_ids)}] {sid}: skipped (no labelled vessels)"); continue
    scene_dir = os.path.join(CONFIG["data_dir"], sid)
    try:
        download_scene_files(sid, url_by_scene[sid], CONFIG["data_dir"])
        S = prepare_scene(df_val, sid, CONFIG)                     # benchmark window, mask, labels, OS background
        r_off, c_off, full_w, full_h, win_tf, crs_wkt = scene_window_offsets(df_val, sid, CONFIG)
        crop = CONFIG["crop_size"]
        vh_db = read_window(os.path.join(scene_dir, "VH_dB.tif"), r_off, c_off, crop)
        vv_db = read_window(os.path.join(scene_dir, "VV_dB.tif"), r_off, c_off, crop)
        with rasterio.open(os.path.join(scene_dir, "owiWindSpeed.tif")) as w:
            sx, sy = w.width / full_w, w.height / full_h
            wind = w.read(1, window=Window(c_off * sx, r_off * sy, crop * sx, crop * sy)).astype(np.float32)
    except Exception as e:
        print(f"[{i}/{len(scene_ids)}] {sid}: FAILED ({str(e)[:150]})")
        shutil.rmtree(scene_dir, ignore_errors=True); continue

    # 1) scene cache (read by later stages instead of downloading again)
    meta = dict(scene_id=sid, r_off=r_off, c_off=c_off, crop=crop, full_w=full_w, full_h=full_h,
                window_transform=list(win_tf)[:6], crs_wkt=crs_wkt, calibration=sid == CONFIG["calibration_scene"])
    np.savez_compressed(os.path.join(CACHE_DIR, f"{sid}.npz"), vh=vh_db.astype(np.float16), vv=vv_db.astype(np.float16),
                        water=S["water_mask"], wind=wind, meta=json.dumps(meta))

    # 2) candidates (OS-CFAR, T = 5.0 dB) and their labels
    os_algo = algo_table(S, CONFIG)["Hollow OS-CFAR"]
    pts = extract_centroids(detect(S, os_algo, CAND_T, CONFIG), S["water_mask"], CONFIG)
    rows = df_val[df_val["scene_id"] == sid]
    eval_df = rows[rows["is_vessel"] == True].copy()
    eval_df["local_row"] = eval_df["detect_scene_row"] - r_off; eval_df["local_col"] = eval_df["detect_scene_column"] - c_off
    eval_df = eval_df[(eval_df["local_row"] >= 0) & (eval_df["local_row"] < crop) & (eval_df["local_col"] >= 0) & (eval_df["local_col"] < crop)]
    ri = eval_df["local_row"].astype(int).clip(0, crop - 1).values; ci = eval_df["local_col"].astype(int).clip(0, crop - 1).values
    eval_df = eval_df[S["water_mask"][ri, ci]].reset_index(drop=True)
    assert len(eval_df) == len(S["gt_all"]) and np.allclose(eval_df[["local_row", "local_col"]].values, S["gt_all"]), "evaluated-vessel list does not match prepare_scene"
    cand_label, reason, src_lab, conf_lab = label_candidates(pts, S, eval_df, CONFIG["match_distance_px"])

    # 3) chips from the unfiltered images
    chips = make_chips(vh_db, vv_db, pts) if len(pts) else np.empty((0, 2, CHIP, CHIP), np.float16)
    np.savez_compressed(os.path.join(CHIPS_DIR, f"{sid}.npz"), chips=chips)
    idx = pd.DataFrame({"scene_id": sid, "cand": np.arange(len(pts)),
                        "row": pts[:, 0] if len(pts) else [], "col": pts[:, 1] if len(pts) else [],
                        "label": cand_label, "exclude_reason": reason, "source": src_lab, "confidence": conf_lab,
                        "calibration_scene": sid == CONFIG["calibration_scene"]})
    idx.to_csv(CHIPS_INDEX, mode="a", header=not os.path.exists(CHIPS_INDEX), index=False)

    # 4) consistency check against the 49-scene benchmark (OS-CFAR, ref T = 5.0)
    tp, fp, fn, *_ = evaluate_predictions(pts, S["gt_all"], S["ig_all"], CONFIG["match_distance_px"])
    check = "no benchmark row"
    if bench is not None:
        b = bench[(bench["scene_id"] == sid) & (bench["detector"] == "Hollow OS-CFAR") & (bench["point"] == "ref")]
        if len(b):
            b = b.iloc[0]
            check = "matches benchmark" if (tp, fp, fn) == (b["tp"], b["fp"], b["fn"]) else f"MISMATCH (benchmark {b['tp']}/{b['fp']}/{b['fn']})"
    shutil.rmtree(scene_dir, ignore_errors=True)
    del S, vh_db, vv_db
    n_pos, n_neg, n_ex = int((cand_label == 1).sum()), int((cand_label == 0).sum()), int((cand_label == -1).sum())
    print(f"[{i}/{len(scene_ids)}] {sid}: {len(pts)} candidates (pos {n_pos}, neg {n_neg}, excluded {n_ex}) | "
          f"OS T=5 TP/FP/FN {tp}/{fp}/{fn} {check} | {time.time() - t0:.0f} s")

print(f"\nDone in {(time.time() - t_start) / 60:.1f} min.")


## 6. Summary

In [ ]:
idx = pd.read_csv(CHIPS_INDEX)
print(f"Scenes with chips: {idx['scene_id'].nunique()} | cache files: {len([f for f in os.listdir(CACHE_DIR) if f.endswith('.npz')])}")
print(idx.groupby("label").size().rename({1: "positive", 0: "negative", -1: "excluded"}).to_string())
print(idx[idx["label"] == -1].groupby("exclude_reason").size().to_string())
print(idx[idx["label"] == 1].groupby("source").size().to_string())
size_mb = sum(os.path.getsize(os.path.join(d, f)) for d in [CACHE_DIR, CHIPS_DIR] for f in os.listdir(d)) / 1e6
print(f"Total size in Drive folder (cache + chips): {size_mb:.0f} MB")
